<a href="https://colab.research.google.com/github/ngost2002/MATH_AI/blob/main/Code/Einstein.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this sheet you will implement an $L_2$ distance metric between two vectors using Einstein summation notation.

We will construct the difference three ways:
1. A direct calculation using loops.
2. Using the norm of the difference of the two vectors.
3. Using Einstein summation notation to construct the dot product.
We are going to time each of the three to see which is best.

Then we will generalise this to work on a tensor consisting of a list of $m$ vectors. We want to calculate the distance between each pair.

In [1]:
import torch
import math

In [2]:
# construct two simple vectors
k = 1000
x1 = torch.arange(1.0,k)
x2 = torch.arange(1.0,k) * x1

In [3]:
# define three ways of doing the distance
def norm1(u, v):
  total = 0.0
  for i in range(u.shape[0]):
    total += ( u[i] - v[i] ) ** 2
  return math.sqrt(total)

def norm2(u, v):
  return torch.linalg.norm( u - v , ord=2).item()

def norm3(u, v):
  dot = torch.einsum('i,i -> ', u, v) # I could have used `torch.dot` here
  tmp = torch.einsum( 'i -> ', u ** 2) + torch.einsum( 'i -> ', v ** 2) - 2*dot
  return math.sqrt(tmp)

# test them
print( norm1(x1,x2) )
print( norm2(x1,x2) )
print( norm3(x1,x2) )

14106797.04423481
14106795.0
14106795.260284455


In [4]:
# measure the times
%time norm1(x1,x2)
%time norm2(x1,x2)
%time norm3(x1,x2)


CPU times: user 16.8 ms, sys: 860 µs, total: 17.7 ms
Wall time: 18.4 ms
CPU times: user 542 µs, sys: 0 ns, total: 542 µs
Wall time: 514 µs
CPU times: user 370 µs, sys: 0 ns, total: 370 µs
Wall time: 376 µs


14106795.260284455

Now we need to think about a set of vectors ${\mathbf x}_1, \ldots, {\mathbf x}_n$. We want to calculate the distances between each pair so there are $n (n-1)/2$ total distance calculations. There are two ways you could calculate these:
1. Perform a loop, and calculate each pair of distances (using the fastest of the three functions above).
2. Use Einstein summation notation to do it in one hit. Note that, the Einstein summation approach wasn't the fastest above, but when we do it this way the norms of each individual vector need only be calculated once, so the only part that need be done for each pair is the dot product.

We will store the set of $n$ vectors in an $n \times k$ matrix (an order 2 tensor). The output should be an $n \times n$ (symmetric) matrix with zeros along the diagonals.

In [21]:
# write two functions

def Norm_iterate( X ):
  n = X.shape[0]
  d = torch.zeros(n, n) # return an nxn matrix
    # use a loop for i=1 to n and j from i+1 to n
  for i in range(n):
    for j in range(i + 1, n):
            d[i, j] = norm1(X[i],X[j])
            d[j, i] = d[i, j]
  return d

def Norm_Einstein( X ):
  n = X.shape[0]
  d = torch.zeros(n, n) # return an nxn matrix
    # use Einstein summation notation
  for i in range(n):
      for j in range(i + 1, n):
            d[i,j]=norm3(X[i],X[j])
            d[j,i]=d[i,j]
  return d


First check that the results from your two functions are correct using a small set of vectors for which you know the results.

In [22]:
A=torch.tensor([[1,2],[3,4]], dtype=torch.float)
print(Norm_iterate(A))
print(Norm_Einstein(A))

tensor([[0.0000, 2.8284],
        [2.8284, 0.0000]])
tensor([[0.0000, 2.8284],
        [2.8284, 0.0000]])


In [29]:
N = 100
k = 200
A = torch.rand(N, k)

In [30]:
%%timeit
a = Norm_iterate(A)

16 s ± 744 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [32]:
%%timeit
a = Norm_Einstein(A)

613 ms ± 58.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Now time your functions and determine which is best on a larger set of vectors.

My answer is Norm_Einstein work much better for larger matrix